# ??????

In [ ]:
import matplotlib.pyplot as plt
import mplfinance as mpf
import numpy as np
import pandas as pd
import yfinance as yf

## ??????? yf.Ticker ?????????

In [ ]:
symbol = '^N225'
ticker = yf.Ticker(symbol)

## ??6???????????

In [ ]:
df = ticker.history(period='6mo')
df

### ???????????????????

Reference: https://mekatana.com/python-momentum/

In [ ]:
# ????????
period = 10
df['momentum'] = df['Close'] - df['Close'].shift(period)

apd_roscillator = [
    mpf.make_addplot(
        df['momentum'],
        ylabel='Momentum',
        color='b',
        width=0.75,
        panel=2,
    )
]

mpf.plot(
    df, type='candle', style='yahoo', mav=(5, 25, 75),
    volume=True,
    addplot=apd_roscillator,
    title=symbol,
    figsize=(12, 6),
    tight_layout=True,
    #savefig=dict(fname='n225_momentum.jpeg', dpi=100)
)

### ???????????????????

Reference: https://mekatana.com/bollinger-bands-python/

In [ ]:
# ?????????
period = 10
r = df['Close'].rolling(period)    
df['upper3'] = r.mean() + 3 * r.std()
df['upper2'] = r.mean() + 2 * r.std()
df['upper1'] = r.mean() + 1 * r.std()
df['mean'] = r.mean()
df['lower1'] = r.mean() - 1 * r.std()
df['lower2'] = r.mean() - 2 * r.std()
df['lower3'] = r.mean() - 3 * r.std()

df_upper3_max=df['upper3'].max() + 10
df_lower3_min=df['lower3'].min() - 10 

apd_oscilator  = [
    mpf.make_addplot(df[['upper3', 'upper2', 'upper1','mean', 'lower1', 'lower2', 'lower3']], width=0.75)
]   

mpf.plot(
    df, type='candle', style='yahoo',
    volume=True,
    title=symbol,
    addplot=apd_oscilator,
    ylim=(df_lower3_min, df_upper3_max),
    figsize=(12, 5),
    tight_layout=True,
    #savefig=dict(fname='n225_bollinger-bands.jpeg', dpi=100)
)

## ??10???????

In [ ]:
df_long = ticker.history(period='10y', interval='1mo')
df_long

### 2020?3????????

????????2017?3?(1?8909?26?)???3??????????????????????????????????????????????????????????????????????????????????????????????????????????

Reference: https://indexes.nikkei.co.jp/atoz/2020/04/202003.html

In [ ]:
origin = pd.to_datetime('2018-01-01 00:00:00+09:00')
price_min = min(df_long[df_long.index > origin]['Low'])
df_long_min = df_long[df_long['Low'] == price_min]
#price_min = df_long_min['Low'].iloc[0]
ts_min = df_long_min.index[0]
print(ts_min, price_min)
df_long_min

### 2024?7????????

7????????????(FRB)???????????????????????????????????????????????????????11???4?2224?02??????????4?2000????????

Reference: https://indexes.nikkei.co.jp/atoz/2024/08/20247.html

In [ ]:
price_max = max(df_long['High'])
df_long_max = df_long[df_long['High'] == price_max]
#price_max = df_long_max['High'].iloc[0]
ts_max = df_long_max.index[0]
print(ts_max, price_max)
df_long_max

In [ ]:
delta = price_max - price_min
level1 = price_max - 0.236 * delta
level2 = price_max - 0.382 * delta
level3 = price_max - 0.618 * delta

## ??????????????

In [ ]:
plt.rcParams['font.family'] = 'monospace'
fig, ax = plt.subplots(figsize=(12, 4))

mpf.plot(
    df_long, type='candle', style='binance', datetime_format='%Y-%m', xrotation=0, mav=(12, 36, 60),
    ax = ax,
)

# ??????????????
x_start = 0.45
x_left = ax.get_xlim()[0]
x_right = ax.get_xlim()[1]
x_mid = (x_right - x_left) * (x_start - 0.05)
x_delta = x_right - x_left
x_adjust = x_delta * 0.005

y_min = ax.get_ylim()[0]
y_max = ax.get_ylim()[1]
y_delta = y_max - y_min
y_adjust = y_delta * 0.01
for y, v in zip([price_max, level1, level2, level3, price_min],[0, 23.6, 38.2, 61.8, 100]):
    plt.axhline(y, xmin=x_start, label = 'Top', linestyle = 'dashed', linewidth=0.75, color = 'b')
    plt.text(x_right + x_adjust, y - y_adjust, '%.1f' % y, color='b')
    plt.text(x_mid, y + y_adjust, '%.1f%%' % v, color='b')

plt.title('Nikkei 225 Monthly Chart\nFibonacci retracement')
plt.savefig('n255_fibonacci.png')
plt.show()